# Final tangent representation: direct updates versus periodic refits

This notebook compares how well two MMNN parameter-update rules represent the high-frequency component of the same two-player non-potential game,

$$
b_\omega(x)=d(x)+q_\omega(x),
$$

where

$$
d(x)=\begin{pmatrix}-\kappa x_1\\-\kappa x_2\end{pmatrix},
\qquad
q_\omega(x)=\begin{pmatrix}A\sin(\omega x_2)\\-A\sin(\omega x_1)\end{pmatrix}.
$$

The comparison uses identical training sample sizes, validation sample sizes, MMNN architectures, initial weights, initial particles, and ordinary-step random tangent schedules. Only the parameter-update rule changes.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np

# Locate DTB_Ver3 locally/on PACE. Clone the branch only in Colab when absent.
candidates = [Path.cwd(), *Path.cwd().parents]
repo_root = next((p for p in candidates if (p / 'DTB_Ver3').is_dir()), None)
if repo_root is None:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Run inside the repository or upload the DTB_Ver3 folder.')
    repo_root = Path('/content/dtb-colab-experiments')
    if not repo_root.exists():
        subprocess.run([
            'git', 'clone', '--depth', '1', '--branch', 'codex/game-dynamics-dtb',
            'https://github.com/sun-mengwei/dtb-colab-experiments.git', str(repo_root),
        ], check=True)
sys.path.insert(0, str(repo_root.resolve()))

from DTB_Ver3 import RepresentationComparisonConfig, run_representation_comparison
from DTB_Ver3.utils import write_csv

output_dir = repo_root / 'DTB_Ver3' / 'results' / 'representation_direct_vs_periodic_refit'
output_dir.mkdir(parents=True, exist_ok=True)
print('repository:', repo_root)
print('output:', output_dir)


## 1. Matched experimental protocol

For the **direct update**, every outer step projects on the scheduled random coordinate set $S_k$ and applies

$$
X_{k+1}=X_k+hJ_{S_k}\alpha_k,
\qquad
\theta_{k+1}[S_k]=\theta_k[S_k]+h\alpha_k.
$$

For the **periodic refit**, ordinary steps use the same scheduled random subsets while keeping $\theta$ fixed. Every 10% of the outer steps, the method uses all trainable tangent directions for the particle projection and then solves the unregularized fit

$$
\theta_{k+1}=\arg\min_\theta
\frac{1}{N}\|T_\theta(z)-X_{k+1}\|_F^2.
$$

A random subset is generated at every index for both methods. At reset indices the periodic method consumes that draw but replaces it with the full basis, so subsequent ordinary-step random subsets remain aligned.

In [ ]:
# Game frequencies and matched Monte Carlo seeds.
OMEGA_MULTIPLIERS = (1.0, 4.0, 8.0, 16.0)
SEEDS = (2026, 2027, 2028)
KAPPA = 1.0
AMPLITUDE = 1.0

# Both methods use exactly these sample counts.
TRAINING_SIZE = 10_000
VALIDATION_SIZE = 10_000
VALIDATION_SEED = 9026

# Outer DTB integration and common refined RK4 references.
STEP_SIZE = 0.001
FINAL_TIME = 0.2
RK4_REFERENCE_STEP = 0.000125

# Identical residual-MMNN architecture for both methods.
MMNN_WIDTH = 12
MMNN_RANK = 12
MMNN_DEPTH = 3
ACTIVATION = 'tanh'
RANDOM_TANGENT_SIZE = 128
SVD_RTOL = 1e-8
JACOBIAN_CHUNK = 512

# Periodic full-basis reset controls.
REFIT_INTERVAL_FRACTION = 0.10
REFIT_LEARNING_RATE = 1e-3
REFIT_MAXIMUM_STEPS = 100
REFIT_RELATIVE_TOLERANCE = 0.05
REFIT_ABSOLUTE_TOLERANCE = 1e-7

DTYPE_NAME = 'float64'
DEVICE_NAME = 'auto'

print({
    'omega_over_pi': OMEGA_MULTIPLIERS,
    'seeds': SEEDS,
    'training_size_per_method': TRAINING_SIZE,
    'validation_size_per_method': VALIDATION_SIZE,
    'outer_steps': int(round(FINAL_TIME / STEP_SIZE)),
    'refit_interval_fraction': REFIT_INTERVAL_FRACTION,
    'MMNN_width_rank_depth': (MMNN_WIDTH, MMNN_RANK, MMNN_DEPTH),
})


## 2. Run the frequency sweep

For every $(\omega,mathrm{seed})$ pair, the package constructs one identity-initialized residual MMNN and copies it for the two methods. Both runs receive the same 10,000 training particles.

An independent set of 10,000 validation labels is evolved to $T$ by refined RK4. Both final tangent spaces are evaluated against the same target $q_\omega(X_{\mathrm{RK4}}(T))$.

In [ ]:
config = RepresentationComparisonConfig(
    omega_multipliers=OMEGA_MULTIPLIERS,
    seeds=SEEDS,
    training_size=TRAINING_SIZE,
    validation_size=VALIDATION_SIZE,
    validation_seed=VALIDATION_SEED,
    kappa=KAPPA,
    amplitude=AMPLITUDE,
    step_size=STEP_SIZE,
    final_time=FINAL_TIME,
    reference_step_size=RK4_REFERENCE_STEP,
    width=MMNN_WIDTH,
    rank=MMNN_RANK,
    depth=MMNN_DEPTH,
    activation=ACTIVATION,
    random_tangent_size=RANDOM_TANGENT_SIZE,
    svd_rtol=SVD_RTOL,
    jacobian_chunk=JACOBIAN_CHUNK,
    refit_interval_fraction=REFIT_INTERVAL_FRACTION,
    refit_learning_rate=REFIT_LEARNING_RATE,
    refit_maximum_steps=REFIT_MAXIMUM_STEPS,
    refit_relative_tolerance=REFIT_RELATIVE_TOLERANCE,
    refit_absolute_tolerance=REFIT_ABSOLUTE_TOLERANCE,
    dtype=DTYPE_NAME,
    device=DEVICE_NAME,
    progress_reports=4,
)

comparison = run_representation_comparison(config)
print({
    'rows': len(comparison.records),
    'trainable_MMNN_coordinates': comparison.parameter_count,
    'training_size': comparison.config.training_size,
    'validation_size': comparison.config.validation_size,
})


## 3. Final representation metric

Let $Z_{\mathrm{ref}}$ be the common independent validation labels and let

$$
X_{\mathrm{ref}}^\omega(T)=\Phi_T^\omega(Z_{\mathrm{ref}})
$$

be their refined RK4 state. For each method $m$, the notebook forms its **full** final tangent matrix

$$
J_m=D_\theta T_{\theta_T^{(m)}}(Z_{\mathrm{ref}})
$$

and projects the common oscillatory target

$$
q_{\omega,\mathrm{ref}}=q_\omega(X_{\mathrm{ref}}^\omega(T)).
$$

The best full-basis coefficient and representation error are

$$
\alpha_{m,\mathrm{ref}}
=\arg\min_\alpha\|J_m\alpha-q_{\omega,\mathrm{ref}}\|_2^2,
$$

$$
E_{\mathrm{repr}}^{(m)}(\omega)
=
\frac{\|J_m\alpha_{m,\mathrm{ref}}-q_{\omega,\mathrm{ref}}\|_2}
{\|q_{\omega,\mathrm{ref}}\|_2}.
$$

Because both final evaluations use all trainable coordinates and the same target, this metric compares the representation power learned by the two update rules rather than their last random subsets.

In [ ]:
raw_csv = write_csv(
    output_dir / 'representation_comparison_raw.csv',
    comparison.columns,
    comparison.records,
)
records = [dict(zip(comparison.columns, row)) for row in comparison.records]
methods = ('direct', 'periodic_refit')

# Aggregate matched seeds for each frequency and method.
summary_rows = []
for multiplier in OMEGA_MULTIPLIERS:
    for method in methods:
        selected = [
            row for row in records
            if row['omega_multiple'] == float(multiplier) and row['method'] == method
        ]
        representation = np.asarray([row['representation_error'] for row in selected])
        captured = np.asarray([row['captured_energy'] for row in selected])
        alpha = np.asarray([row['alpha_norm'] for row in selected])
        rank = np.asarray([row['retained_rank'] for row in selected], dtype=float)
        condition = np.asarray([row['condition_number'] for row in selected])
        trajectory = np.asarray([row['final_trajectory_rms'] for row in selected])
        ddof = 1 if len(selected) > 1 else 0
        summary_rows.append((
            float(multiplier), method, len(selected),
            representation.mean(), representation.std(ddof=ddof),
            captured.mean(), captured.std(ddof=ddof),
            alpha.mean(), alpha.std(ddof=ddof),
            rank.mean(), rank.std(ddof=ddof),
            condition.mean(), condition.std(ddof=ddof),
            trajectory.mean(), trajectory.std(ddof=ddof),
        ))

summary_columns = (
    'omega_multiple', 'method', 'seed_count',
    'representation_error_mean', 'representation_error_std',
    'captured_energy_mean', 'captured_energy_std',
    'alpha_norm_mean', 'alpha_norm_std',
    'retained_rank_mean', 'retained_rank_std',
    'condition_number_mean', 'condition_number_std',
    'final_trajectory_rms_mean', 'final_trajectory_rms_std',
)
summary_csv = write_csv(
    output_dir / 'representation_comparison_summary.csv',
    summary_columns,
    summary_rows,
)

# Compute the periodic/direct representation-error ratio using paired seeds.
ratio_rows = []
for multiplier in OMEGA_MULTIPLIERS:
    ratios = []
    for seed in SEEDS:
        direct = next(
            row for row in records
            if row['omega_multiple'] == float(multiplier)
            and row['seed'] == seed and row['method'] == 'direct'
        )
        periodic = next(
            row for row in records
            if row['omega_multiple'] == float(multiplier)
            and row['seed'] == seed and row['method'] == 'periodic_refit'
        )
        ratio = periodic['representation_error'] / direct['representation_error']
        ratios.append(ratio)
        ratio_rows.append((float(multiplier), seed, ratio))
ratio_csv = write_csv(
    output_dir / 'periodic_to_direct_error_ratio.csv',
    ('omega_multiple', 'seed', 'periodic_to_direct_ratio'),
    ratio_rows,
)
print('saved:', raw_csv)
print('saved:', summary_csv)
print('saved:', ratio_csv)


## 4. Representation comparison plots

Values below one in the periodic/direct ratio mean that periodic full-basis refitting produced the more expressive final tangent space for the common oscillatory target.

In [ ]:
colors = {'direct': 'tab:blue', 'periodic_refit': 'tab:orange'}
labels = {'direct': 'direct parameter update', 'periodic_refit': 'periodic full-basis refit'}
summary = [dict(zip(summary_columns, row)) for row in summary_rows]

fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
metric_specs = (
    ('representation_error_mean', 'representation_error_std', r'$E_{repr}$', True),
    ('captured_energy_mean', 'captured_energy_std', r'$1-E_{repr}^2$', False),
    ('condition_number_mean', 'condition_number_std', 'condition number', True),
    ('final_trajectory_rms_mean', 'final_trajectory_rms_std', 'final DTB/RK4 RMS', True),
)
for axis, (mean_key, std_key, ylabel, log_scale) in zip(axes.flat, metric_specs):
    for method in methods:
        method_rows = [row for row in summary if row['method'] == method]
        x = np.asarray([row['omega_multiple'] for row in method_rows])
        mean = np.asarray([row[mean_key] for row in method_rows])
        std = np.asarray([row[std_key] for row in method_rows])
        axis.errorbar(
            x, mean, yerr=std, marker='o', capsize=3,
            color=colors[method], label=labels[method],
        )
    if log_scale:
        axis.set_yscale('log')
    axis.set(xlabel=r'$\omega/\pi$', ylabel=ylabel, title=f'{ylabel} versus frequency')
    axis.set_xticks(OMEGA_MULTIPLIERS)
    axis.grid(True, which='both', alpha=0.3)
    axis.legend()
comparison_plot = output_dir / 'representation_comparison.png'
fig.savefig(comparison_plot, dpi=180, bbox_inches='tight')
plt.show()

fig, axis = plt.subplots(figsize=(8, 4.2), constrained_layout=True)
ratio_means = []
ratio_stds = []
for multiplier in OMEGA_MULTIPLIERS:
    values = np.asarray([row[2] for row in ratio_rows if row[0] == float(multiplier)])
    ratio_means.append(values.mean())
    ratio_stds.append(values.std(ddof=1 if len(values) > 1 else 0))
axis.errorbar(
    OMEGA_MULTIPLIERS, ratio_means, yerr=ratio_stds,
    marker='o', capsize=3, color='tab:purple',
)
axis.axhline(1.0, color='black', linestyle='--', label='equal representation error')
axis.set(
    xlabel=r'$\omega/\pi$',
    ylabel=r'$E_{repr}^{periodic}/E_{repr}^{direct}$',
    title='Matched representation-error ratio',
    xticks=OMEGA_MULTIPLIERS,
)
axis.grid(True, alpha=0.3)
axis.legend()
ratio_plot = output_dir / 'periodic_to_direct_representation_ratio.png'
fig.savefig(ratio_plot, dpi=180, bbox_inches='tight')
plt.show()
print('saved:', comparison_plot)
print('saved:', ratio_plot)


## 5. Interpretation

- If periodic refitting gives a substantially smaller $E_{\mathrm{repr}}$ at large $\omega$, the resets have adapted the final tangent space to high-frequency directions that direct parameter Euler updates did not retain.
- If both errors increase similarly with $\omega$, the shared MMNN architecture is likely the limiting factor.
- If $E_{\mathrm{repr}}$ is small but the trajectory RMS remains large, representation is adequate and the failure is more consistent with time discretization or dynamical error amplification.
- A lower error accompanied by an extreme condition number or coefficient norm indicates that the oscillatory target is represented through weak tangent directions and may be numerically fragile.

In [ ]:
archive_path = Path(shutil.make_archive(
    str(output_dir),
    'zip',
    root_dir=output_dir,
))
print('result archive:', archive_path)
